# Naturvårdsverket Friluftsliv → OSM-granskning

Syftet är **inte** att anta att Naturvårdsverkets objekt redan är länkade data.

Notebooken gör i stället en praktisk granskningsyta:

1. Hämtar `Leder.geojson`, `Anordningar.geojson`, `Publiceringsstatus.geojson` och `Statliga_Leder.geojson`.
2. Behåller **alla metadatafält** från Naturvårdsverket.
3. Transformerar koordinater till WGS84.
4. Skapar en **klickbar OSM-länk** för varje objekt baserad på dess koordinat.
5. Skapar klickbara HTML-tabeller så att du kan gå igenom objekten och kontrollera OSM manuellt.
6. Skapar en interaktiv karta med popup för varje objekt. Popupen innehåller en direktlänk till OSM vid objektets position.
7. Räknar objekt per typ/undertyp/kategori där sådana fält finns.

**Viktigt:** En OSM-länk betyder bara att vi öppnar OSM på objektets geografiska position. Notebooken påstår inte att objektet finns i OSM förrän du faktiskt har kontrollerat det.

In [1]:
import io
import json
import re
import requests
import pandas as pd
import geopandas as gpd
from IPython.display import display, HTML

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 120)

BASE = 'https://geodata.naturvardsverket.se/nedladdning/friluftsliv/'
FILES = {
    'Leder': 'Leder.geojson',
    'Anordningar': 'Anordningar.geojson',
    'Publiceringsstatus': 'Publiceringsstatus.geojson',
    'Statliga_Leder': 'Statliga_Leder.geojson',
}

print('Naturvårdsverkets friluftslivsdata')
print(BASE)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


Naturvårdsverkets friluftslivsdata
https://geodata.naturvardsverket.se/nedladdning/friluftsliv/


## 1. Läs in data

GeoJSON-filerna är stora, så de hämtas direkt från Naturvårdsverkets officiella nedladdningskatalog. Katalogen innehåller de fyra aktuella filerna. citeturn0search0

In [2]:
layers = {}

for name, filename in FILES.items():
    url = BASE + filename
    print(f'Läser {name}: {url}')
    layers[name] = gpd.read_file(url)
    print(f'  {len(layers[name]):,} objekt, CRS={layers[name].crs}')

print('\nKlart.')

Läser Leder: https://geodata.naturvardsverket.se/nedladdning/friluftsliv/Leder.geojson
  12,011 objekt, CRS=EPSG:3006
Läser Anordningar: https://geodata.naturvardsverket.se/nedladdning/friluftsliv/Anordningar.geojson
  21,579 objekt, CRS=EPSG:3006
Läser Publiceringsstatus: https://geodata.naturvardsverket.se/nedladdning/friluftsliv/Publiceringsstatus.geojson
  8,790 objekt, CRS=EPSG:3006
Läser Statliga_Leder: https://geodata.naturvardsverket.se/nedladdning/friluftsliv/Statliga_Leder.geojson
  2,004 objekt, CRS=EPSG:3006

Klart.


In [3]:
def ensure_wgs84(gdf):
    if gdf.crs is None:
        # Naturvårdsverkets friluftslivsdata levereras normalt i SWEREF 99 TM.
        # Sätt bara CRS om det saknas; transformera sedan till WGS84.
        gdf = gdf.set_crs(3006)
    return gdf.to_crs(4326)

wgs84_layers = {name: ensure_wgs84(gdf.copy()) for name, gdf in layers.items()}

for name, gdf in wgs84_layers.items():
    print(name, '→', gdf.crs)

Leder → EPSG:4326
Anordningar → EPSG:4326
Publiceringsstatus → EPSG:4326
Statliga_Leder → EPSG:4326


## 2. Skapa en punkt för OSM-granskning

För linjer används centroiden som granskningspunkt. För polygoner används en representativ punkt. Detta gör att även leder kan få en OSM-länk.

Länken öppnar OSM exakt i närheten av Naturvårdsverkets objekt. Det är avsiktligt en **manuell verifieringslänk**, inte ett påstående om att OSM har samma objekt.

In [4]:
def add_review_columns(gdf):
    out = gdf.copy()
    review_points = out.geometry.representative_point()
    out['latitude_wgs84'] = review_points.y.round(7)
    out['longitude_wgs84'] = review_points.x.round(7)
    out['osm_link'] = [
        f'https://www.openstreetmap.org/?mlat={lat}&mlon={lon}#map=18/{lat}/{lon}'
        for lat, lon in zip(out['latitude_wgs84'], out['longitude_wgs84'])
    ]
    # Overpass-länk: användbar för att snabbt se OSM-objekt inom ca 50 m.
    out['osm_nearby_link'] = [
        'https://overpass-turbo.eu/?Q=' +
        f'[out:json];(nwr(around:50,{lat},{lon}););out center;'
        for lat, lon in zip(out['latitude_wgs84'], out['longitude_wgs84'])
    ]
    return out

review_layers = {name: add_review_columns(gdf) for name, gdf in wgs84_layers.items()}

for name, gdf in review_layers.items():
    print(name, 'har nu', len(gdf), 'granskningsobjekt')

Leder har nu 12011 granskningsobjekt
Anordningar har nu 21579 granskningsobjekt
Publiceringsstatus har nu 8790 granskningsobjekt
Statliga_Leder har nu 2004 granskningsobjekt


## 3. Identifiera metadatafält

Här visas samtliga attributfält. Det är medvetet: vi vill först se **vad Naturvårdsverket faktiskt publicerar**, inte gissa vilka fält som är viktiga.

In [5]:
field_rows = []
for name, gdf in review_layers.items():
    for col in gdf.columns:
        if col != 'geometry':
            field_rows.append({
                'lager': name,
                'fält': col,
                'dtype': str(gdf[col].dtype),
                'ifyllda': int(gdf[col].notna().sum()),
                'unika': int(gdf[col].nunique(dropna=True)),
            })

fields_df = pd.DataFrame(field_rows)
display(fields_df.sort_values(['lager', 'fält']).reset_index(drop=True))

,lager,fält,dtype,ifyllda,unika
0,Anordningar,ANORDNINGNAMN,object,9562,5707
1,Anordningar,ANORDNING_ID,object,21579,21550
2,Anordningar,BESKRIVNING,object,2632,1870
3,Anordningar,GEOMETRIKVALITET,object,21474,6
4,Anordningar,NP,int32,21579,2
5,Anordningar,OBJECTID,int32,21579,21579
6,Anordningar,SKYDDATOMRADE,object,21024,4232
7,Anordningar,SKYDDATOMRADE_ID,object,21024,4232
8,Anordningar,SKYDDSTYP_KOD,object,21024,43
9,Anordningar,STATLIGLED,object,976,234


## 4. Kategorier: typ / undertyp / kategori

Notebooken letar efter vanliga namn på klassificeringsfält och visar antal objekt. Detta ger en snabb bild av vad som faktiskt finns i datat.

In [6]:
TYPE_PATTERNS = [
    r'^typ$', r'undertyp', r'kategori', r'^type$', r'subtype', r'category',
    r'ledtyp', r'ledkategori', r'anordning', r'objekttyp'
]

def candidate_type_columns(columns):
    result = []
    for c in columns:
        if c == 'geometry':
            continue
        if any(re.search(p, c, flags=re.I) for p in TYPE_PATTERNS):
            result.append(c)
    return result

for name, gdf in review_layers.items():
    cols = candidate_type_columns(gdf.columns)
    print(f'\n### {name}')
    print('Möjliga klassificeringsfält:', cols)
    for col in cols:
        print(f'\n{col}:')
        display(gdf[col].fillna('(saknas)').value_counts(dropna=False).rename_axis(col).reset_index(name='antal').head(100))


### Leder
Möjliga klassificeringsfält: ['LEDTYP', 'LEDKATEGORI']

LEDTYP:


,LEDTYP,antal
0,Vandringsled,9048
1,"Skidled, Skoterled",690
2,Skidled,428
3,Omarkerad stig,343
4,Naturstig,245
5,Skoterled,244
6,"Naturstig, Vandringsled",194
7,"Skidled, Vinterled",164
8,"Skoterled, Vinterled",115
9,Vinterled,113



LEDKATEGORI:


,LEDKATEGORI,antal
0,Barmarksled,10175
1,Led på snö,1808
2,Led på/i vatten,28



### Anordningar
Möjliga klassificeringsfält: ['ANORDNING_ID', 'ANORDNINGNAMN', 'TYP', 'UNDERTYP']

ANORDNING_ID:


...


TYP:


,TYP,antal
0,Information,8443
1,Parkering,3021
2,Rastplats,2872
3,Eldstad,1571
4,Dass,1080
...,...,...
56,Pir,2
57,Pulkabacke,2
58,Tillgänglighetsramp,1
59,Barnens skog,1



UNDERTYP:


,UNDERTYP,antal
0,Områdesskyddsinformation,7135
1,Parkering,3021
2,Bänkbord,1625
3,Eldstad,1571
4,Dass,1080
...,...,...
70,Kajakramp,2
71,Laddningsstation för elbilar,2
72,Barnens skog,1
73,Tillgänglighetsramp,1



### Publiceringsstatus
Möjliga klassificeringsfält: []

### Statliga_Leder
Möjliga klassificeringsfält: []


## 5. Klickbar gransknings-tabell

Detta är huvudverktyget för din idé: välj ett lager och få en tabell där varje rad har **OSM** och **OSM nearby**.

- **OSM** = öppnar kartan på objektets position.
- **OSM nearby** = öppnar Overpass Turbo med objekt inom ungefär 50 meter, vilket är användbart när vi vill undersöka om motsvarande OSM-objekt faktiskt finns.

In [7]:
def clickable_table(name, max_rows=500):
    gdf = review_layers[name].copy().head(max_rows)
    display_cols = []
    preferred = [
        'OBJECTID', 'LED_ID', 'LEDNAMN', 'LEDTYP', 'LEDKATEGORI',
        'ANORDNINGSTYP', 'ANORDNING', 'TYP', 'UNDERTYP', 'KATEGORI',
        'latitude_wgs84', 'longitude_wgs84'
    ]
    for c in preferred:
        if c in gdf.columns and c not in display_cols:
            display_cols.append(c)
    display_cols += [c for c in gdf.columns if c not in display_cols and c not in ['geometry', 'osm_link', 'osm_nearby_link']][:10]
    display_cols += ['osm_link', 'osm_nearby_link']
    display_cols = list(dict.fromkeys([c for c in display_cols if c in gdf.columns]))

    html = gdf[display_cols].to_html(index=False, escape=True, render_links=False)
    # Gör de två URL-kolumnerna klickbara.
    for col in ['osm_link', 'osm_nearby_link']:
        pattern = rf'(<td>)({re.escape("https://")})'
    html_df = gdf[display_cols].copy()
    html_df['osm_link'] = html_df['osm_link'].map(lambda x: f'<a href="{x}" target="_blank">OSM</a>')
    html_df['osm_nearby_link'] = html_df['osm_nearby_link'].map(lambda x: f'<a href="{x}" target="_blank">OSM nearby</a>')
    display(HTML(html_df.to_html(index=False, escape=False)))

# Exempel:
clickable_table('Anordningar', max_rows=100)

OBJECTID,TYP,UNDERTYP,latitude_wgs84,longitude_wgs84,ANORDNING_ID,ANORDNINGNAMN,BESKRIVNING,NP,GEOMETRIKVALITET,SKYDDSTYP_KOD,SKYDDATOMRADE,SKYDDATOMRADE_ID,STATLIGLED,STATLIGLED_ID,osm_link,osm_nearby_link
33776721,Fyr,Fyr,55.664043,14.277837,30335917,None,None,1,<= 20 meter,NP,Stenshuvud (2001830),2001830,None,None,OSM,OSM nearby
33788626,Fyr,Fyr,56.300967,12.451604,30474194,None,None,0,<= 20 meter,"N2000-SPA/SCI, NR","Kullaberg (SE0430092), Västra Kullaberg (2000972)","SE0430092, 2000972",None,None,OSM,OSM nearby
33781407,Fyr,Fyr,56.450637,12.542535,30467932,None,None,0,<= 20 meter,"DVO, N2000-SPA/SCI, NR","Bjärehalvöns kuster (2031810), Hallands Väderö (2000962), Hallands Väderö (SE0420002)","2031810, 2000962, SE0420002",None,None,OSM,OSM nearby
33778187,Vägbom,Vägbom,59.297888,13.296044,30208827,Vägbom med kodlås,"Bomen får passeras för transport av personer med fysisk funktionsnedsättning. Ring Länsstyrelsens växel 010-224 70 00 för bomkod. Uppge namn, registreringsnummer och ärende.",0,<= 20 meter,NR,Segerstads skärgård (2002165),2002165,None,None,OSM,OSM nearby
33785368,Vägbom,Vägbom,55.602717,14.198595,30570893,None,None,0,<= 20 meter,"N2000-SCI, NR","Gyllebo (2014529), Gyllebosjön (SE0420305)","2014529, SE0420305",None,None,OSM,OSM nearby
33772787,Vägbom,Vägbom,58.271699,16.261009,30562476,None,None,0,<= 20 meter,NR,Hästenäs kyrkskog (2004572),2004572,None,None,OSM,OSM nearby
33788228,Vägbom,Vägbom,58.673347,16.120040,30562364,None,None,0,<= 20 meter,NR,Ågelsjön (2043608),2043608,None,None,OSM,OSM nearby
33789997,Vägbom,Vägbom,58.810788,17.344025,30557502,None,None,0,Okänd noggrannhet,NR,Nynäs (2001921),2001921,None,None,OSM,OSM nearby
33781659,Vägbom,Vägbom,60.856267,12.850427,30540649,Stängd vägbom Vålhallberget,Vägbommen är stängd och låst.,0,<= 20 meter,NR,Vålhallberget (2014561),2014561,None,None,OSM,OSM nearby
33785038,Vägbom,Vägbom,60.301296,13.273160,30540645,"Vägbom Tijärnsskogen, Dörrfjället",Vägbommen är stängd i tjällossningstid.,0,>20-50 meter,"NR, NR","Dörrfjället (2042566), Titjärnsskogen (2002123)","2042566, 2002123",None,None,OSM,OSM nearby


## 6. Interaktiv karta

Kartan använder Folium. Klicka på ett objekt så visas metadata och en **OSM-länk**. För stora lager kan kartan ta en stund att rendera; använd filtreringen i nästa avsnitt för att prioritera.

In [8]:
!pip -q install folium

import folium

def make_popup(row, layer_name):
    preferred = [
        'OBJECTID', 'LED_ID', 'LEDNAMN', 'LEDTYP', 'LEDKATEGORI',
        'ANORDNINGSTYP', 'ANORDNING', 'TYP', 'UNDERTYP', 'KATEGORI',
        'latitude_wgs84', 'longitude_wgs84'
    ]
    lines = [f'<b>{layer_name}</b>']
    for c in preferred:
        if c in row.index and pd.notna(row[c]):
            lines.append(f'<b>{c}</b>: {str(row[c])}')
    lines.append('<hr>')
    lines.append(f'<a href="{row["osm_link"]}" target="_blank">🗺️ Öppna i OpenStreetMap</a>')
    lines.append(f'<br><a href="{row["osm_nearby_link"]}" target="_blank">🔎 Sök OSM-objekt inom 50 m</a>')
    return folium.Popup('<br>'.join(lines), max_width=400)

def make_map(layer_name, max_objects=3000):
    gdf = review_layers[layer_name]
    # Punktlager används direkt; linjer/polygoner representeras med granskningspunkten.
    pts = gdf.head(max_objects)
    if len(pts) == 0:
        return folium.Map(location=[62, 15], zoom_start=5)

    center = [pts['latitude_wgs84'].mean(), pts['longitude_wgs84'].mean()]
    m = folium.Map(location=center, zoom_start=6, control_scale=True)
    fg = folium.FeatureGroup(name=layer_name, show=True)

    for _, row in pts.iterrows():
        folium.CircleMarker(
            location=[row['latitude_wgs84'], row['longitude_wgs84']],
            radius=4,
            weight=1,
            fill=True,
            popup=make_popup(row, layer_name),
            tooltip=str(row.get('LEDNAMN', row.get('TYP', layer_name)))[:80]
        ).add_to(fg)

    fg.add_to(m)
    folium.LayerControl().add_to(m)
    return m

# Exempel: öppna kartan för Anordningar
make_map('Anordningar', max_objects=3000)

## 7. Prioriteringslista för manuell OSM-kontroll

Här skapar vi en enkel arbetslista. Tanken är att du kan börja med en viss typ/undertyp, sortera på område och sedan klicka igenom OSM-länkarna.

Detta är medvetet **inte automatisk OSM-matchning**. Först gör vi den manuella kontrollen och ser hur väl Naturvårdsverkets objekt faktiskt motsvaras av OSM.

In [9]:
def priority_table(layer_name, type_column=None, value=None, max_rows=500):
    gdf = review_layers[layer_name].copy()
    if type_column and type_column in gdf.columns and value is not None:
        gdf = gdf[gdf[type_column].fillna('').astype(str) == str(value)]

    cols = [c for c in [
        'OBJECTID', 'LED_ID', 'LEDNAMN', 'LEDTYP', 'LEDKATEGORI',
        'TYP', 'UNDERTYP', 'KATEGORI', 'latitude_wgs84', 'longitude_wgs84',
        'osm_link', 'osm_nearby_link'
    ] if c in gdf.columns]
    result = gdf[cols].head(max_rows).copy()
    for c, label in [('osm_link','OSM'), ('osm_nearby_link','OSM nearby')]:
        if c in result:
            result[c] = result[c].map(lambda x, label=label: f'<a href="{x}" target="_blank">{label}</a>')
    display(HTML(result.to_html(index=False, escape=False)))
    return gdf

# Exempel:
# priority_table('Anordningar', type_column='TYP', value='Bänkbord')

## 8. Spara granskningsdata

Alla lager sparas som WGS84-GeoJSON och CSV. CSV-filerna innehåller OSM-länkarna och alla ursprungliga attribut.

In [10]:
OUT = 'naturvardsverket_friluftsliv_output'
import os
os.makedirs(OUT, exist_ok=True)

for name, gdf in review_layers.items():
    geojson_path = os.path.join(OUT, f'{name}_WGS84.geojson')
    csv_path = os.path.join(OUT, f'{name}_granskning.csv')
    gdf.to_file(geojson_path, driver='GeoJSON')
    gdf.drop(columns='geometry').to_csv(csv_path, index=False, encoding='utf-8-sig')
    print('sparat:', geojson_path)
    print('sparat:', csv_path)

sparat: naturvardsverket_friluftsliv_output/Leder_WGS84.geojson
sparat: naturvardsverket_friluftsliv_output/Leder_granskning.csv
sparat: naturvardsverket_friluftsliv_output/Anordningar_WGS84.geojson
sparat: naturvardsverket_friluftsliv_output/Anordningar_granskning.csv
sparat: naturvardsverket_friluftsliv_output/Publiceringsstatus_WGS84.geojson
sparat: naturvardsverket_friluftsliv_output/Publiceringsstatus_granskning.csv
sparat: naturvardsverket_friluftsliv_output/Statliga_Leder_WGS84.geojson
sparat: naturvardsverket_friluftsliv_output/Statliga_Leder_granskning.csv


## Nästa steg: faktisk OSM-matchning

När vi har tittat på några hundra objekt manuellt kan vi använda resultatet för att bygga en riktig matchningsmodell, exempelvis:

- avstånd till OSM-objekt,
- OSM-taggar (`amenity`, `tourism`, `highway`, etc.),
- namnlikhet,
- typ/undertyp,
- geometriöverlapp för leder,
- Wikidata-länk när sådan faktiskt finns.

Då kan vi skapa statusar som **`OSM matchad` / `OSM möjlig match` / `OSM saknas` / `behöver kontroll`** i stället för att bara anta att en länk innebär en koppling.